In [ ]:
# ==============================================================================
# TELJES SCRIPT: ADATBETÖLTÉS + TOKENIZÁLÁS + STABILIZÁLT TANÍTÁS (T5)
# ==============================================================================

# 1. TELEPÍTÉSEK ÉS IMPORTOK
print("Csomagok telepítése és importálás...")
!pip install transformers datasets accelerate sentencepiece evaluate rouge_score bert_score -q
!pip install comet_ml opik -q

import os
import comet_ml
import ast
import torch
import gc
import evaluate
import numpy as np
import pandas as pd
import random
import nltk
from datasets import Dataset
from transformers import (
    T5ForConditionalGeneration,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

os.environ["COMET_API_KEY"] = ""
os.environ["COMET_PROJECT_NAME"] = ""

# NLTK adatok
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# GPU ellenőrzés
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Használt eszköz: {device}")

# Memória tisztítás
torch.cuda.empty_cache()
gc.collect()

# ==============================================================================
# 2. KONFIGURÁCIÓ
# ==============================================================================
MODEL_NAME = "google/flan-t5-base"
RECIPES_PATH = ""
OUTPUT_DIR = "" 

MAX_INPUT_LEN = 256
MAX_OUTPUT_LEN = 512
BATCH_SIZE = 8          
GRAD_ACCUMULATION = 2   
LEARNING_RATE = 3e-4    
EPOCHS = 4              

# ==============================================================================
# 3. ADATBETÖLTÉS ÉS TISZTÍTÁS
# ==============================================================================
print("Adatok betöltése...")
df = pd.read_csv(RECIPES_PATH)

# String -> List konverzió
df["ingredients"] = df["ingredients"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["steps"] = df["steps"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Szűrés
df = df[df['steps'].map(len) >= 4] 
df = df[df['ingredients'].map(len) >= 4]
df = df[df['steps'].map(len) <= 20]

# Mintavételezés
if len(df) > 30000:
    df = df.sample(30000, random_state=42)
    print("Minta lecsökkentve 30.000 minőségi receptre.")

dataset = Dataset.from_pandas(df)
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
test_dataset = dataset_split["test"]
train_val_split = dataset_split["train"].train_test_split(test_size=0.11, seed=42)
train_dataset = train_val_split["train"]
eval_dataset = train_val_split["test"]

print(f"Train: {len(train_dataset)}, Val: {len(eval_dataset)}, Test: {len(test_dataset)}")

# ==============================================================================
# 4. SZÖVEG FORMÁZÁS 
# ==============================================================================
all_ingredients = set()
for ing_list in df["ingredients"]:
    all_ingredients.update(ing_list)
all_ingredients_list = list(all_ingredients)

def format_text_robust(row):
    real_ingredients = row["ingredients"]

    # Zaj generálása 
    num_distractors = random.randint(1, 2)
    distractors = []
    for _ in range(10):
        if len(distractors) >= num_distractors: break
        candidate = random.choice(all_ingredients_list)
        if candidate not in real_ingredients:
            distractors.append(candidate)

    input_list = real_ingredients + distractors
    random.shuffle(input_list)

    # Prompt és Target
    input_text = "generate structured recipe from available ingredients: " + ", ".join(input_list)

    target_text = (
        f"title: {str(row['name']).lower()} "
        f"time: {str(row['minutes'])} mins "
        f"ingredients: {', '.join(real_ingredients).lower()} "
        f"steps: {'; '.join(row['steps']).lower()}"
    )
    return {"input_text": input_text, "target_text": target_text}

print("Szövegek formázása...")
train_dataset = train_dataset.map(format_text_robust)
eval_dataset = eval_dataset.map(format_text_robust)

# ==============================================================================
# 5. TOKENIZÁLÁS
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(batch):
    model_inputs = tokenizer(batch["input_text"], max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(text_target=batch["target_text"], max_length=MAX_OUTPUT_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizálás...")
cols = train_dataset.column_names
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=cols)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True, remove_columns=cols)

# ==============================================================================
# 6. TANÍTÁS (STABILIZÁLT PARAMÉTEREKKEL)
# ==============================================================================
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    # -100 kezelése
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE előkészítés
    decoded_preds = ["\n".join(nltk.sent_tokenize(p.strip())) for p in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(l.strip())) for l in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model.config.use_cache = False

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE, # 3e-4
    warmup_ratio=0.05,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    max_grad_norm=1.0,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=EPOCHS,
    predict_with_generate=True,
    fp16=False,                 # STABILITÁS MIATT KIKAPCSOLVA!
    report_to="comet_ml",
    logging_steps=10,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("\n--- TANÍTÁS INDÍTÁSA ---")
print("Ez a folyamat 20-40 percig tarthat. Kérlek, ne zárd be az ablakot!")
trainer.train()

print("Modell mentése...")
trainer.save_model(os.path.join(OUTPUT_DIR, "final_model"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "final_model"))
print("KÉSZ! Most már jöhet a tesztelés.")
comet_ml.end()

In [ ]:
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

# ==========================================
# 1. BEÁLLÍTÁSOK ÉS ELÉRHETŐSÉG
# ==========================================
# Pontosan az a mappa, ahová a Trainer a végén mentett!
MODEL_PATH = ""
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Használt eszköz: {DEVICE}")
print("Flan-T5 modell és Tokenizer betöltése...")

# ==========================================
# 2. BETÖLTÉS (Hugging Face módszer)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH).to(DEVICE)
model.eval()
print("Flan-T5 sikeresen betöltve!\n")

# ==========================================
# 3. GENERÁLÓ FÜGGVÉNY
# ==========================================
def generate_t5_recipe(ingredients_list, temperature=0.7, max_length=512):
    # FONTOS: A T5-nek meg kell adni a parancsot, amivel tanítottuk!
    prompt = "generate structured recipe from available ingredients: " + ", ".join(ingredients_list).lower()

    # Szöveg tokenizálása
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # Generálás a Hugging Face beépített funkciójával
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            do_sample=True,        # Kreativitás bekapcsolása
            # top_k=50,
            top_p=0.95,
            repetition_penalty=1.2 # Enyhe büntetés, ha ismételné a szavakat
        )

    # Visszafejtés szöveggé
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Szöveg feldarabolása (pont úgy, ahogy a saját modellednél is csináltuk)
    recipe_data = {"title": "N/A", "time": "N/A", "ingredients": [], "steps": "N/A", "raw_text": full_text}
    try:
        recipe_data["title"] = full_text.split("title:")[1].split("time:")[0].strip()
        recipe_data["time"] = full_text.split("time:")[1].split("ingredients:")[0].strip()
        recipe_data["ingredients"] = full_text.split("ingredients:")[1].split("steps:")[0].strip()
        recipe_data["steps"] = full_text.split("steps:")[1].strip()
    except:
        pass

    return recipe_data

# ==========================================
# 4. TESZTELÉS (A "Bolognai-teszt")
# ==========================================
if __name__ == "__main__":
    test_ingredients = ['chicken','rocks', 'rice']

    print(f"Bemeneti hozzávalók: {test_ingredients}")
    print("Recept generálása (ez nagyon gyors lesz)...\n")

    eredmeny = generate_t5_recipe(test_ingredients, temperature=0.7)

    print("="*50)
    print("🍽️ GENERÁLT RECEPT (Flan-T5) 🍽️")
    print("="*50)
    print(f"📌 Cím:        {eredmeny['title']}")
    print(f"⏱️  Idő:        {eredmeny['time']}")
    print(f"🛒 Hozzávalók: {eredmeny['ingredients']}")
    print(f"👩‍🍳 Lépések:    {eredmeny['steps']}")
    print("="*50)


In [ ]:
# ==============================================================================
# TELJES KIÉRTÉKELŐ SCRIPT (ÚJ COLAB PROJEKTHEZ)
# ==============================================================================

# 1. TELEPÍTÉSEK (Mivel új a környezet)
print("Környezet előkészítése...")
!pip install transformers datasets accelerate sentencepiece evaluate rouge_score bert_score -q

import os
import ast
import torch
import gc
import evaluate
import numpy as np
import pandas as pd
import random
import re
import nltk
from tqdm import tqdm
from datasets import Dataset
from transformers import T5ForConditionalGeneration, AutoTokenizer

# NLTK és GPU beállítás
nltk.download("punkt", quiet=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Használt eszköz: {device}")

# ==============================================================================
# 2. MODELL ÉS TOKENIZER BETÖLTÉSE (A DRIVE-RÓL)
# ==============================================================================
# FONTOS: Ellenőrizd, hogy ez az útvonal létezik-e a Drive-odon!
# Az előző kód ide mentett: .../finetuned_v4_FINAL/final_model
MODEL_PATH = ""

print(f"Modell betöltése innen: {MODEL_PATH}")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)
    print("Sikeres betöltés!")
except OSError:
    print("HIBA: Nem találom a modellt! Ellenőrizd, hogy csatoltad-e a Google Drive-ot a bal oldali mappa ikonnal!")
    raise

# ==============================================================================
# 3. TESZT ADATOK ELŐKÉSZÍTÉSE (Hogy legyen mit mérni)
# ==============================================================================
RECIPES_PATH = ""
print("Adatok betöltése a teszthez...")

df = pd.read_csv(RECIPES_PATH)
# Konverziók és tisztítás (ugyanaz, mint tanításnál)
df["ingredients"] = df["ingredients"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["steps"] = df["steps"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df = df[df['steps'].map(len) >= 4]
df = df[df['ingredients'].map(len) >= 4]
df = df[df['steps'].map(len) <= 20]

# Mintavételezés (Ugyanazzal a random_state-tel, hogy ugyanazt a teszthalmazt kapjuk!)
if len(df) > 30000:
    df = df.sample(30000, random_state=42)

dataset = Dataset.from_pandas(df)
# Split (Ugyanaz a seed fontos!)
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
test_dataset = dataset_split["test"]

# Formázó függvény (csak a formátum miatt, zaj nélkül is mehetne, de így konzisztens)
def format_text_for_test(row):
    real_ingredients = row["ingredients"]
    # Itt most nem adunk hozzá extra zajt a méréshez, vagy csak minimálisat
    input_text = "generate structured recipe from available ingredients: " + ", ".join(real_ingredients)
    target_text = (
        f"title: {str(row['name']).lower()} "
        f"time: {str(row['minutes'])} mins "
        f"ingredients: {', '.join(real_ingredients).lower()} "
        f"steps: {'; '.join(row['steps']).lower()}"
    )
    return {"input_text": input_text, "target_text": target_text}

print("Teszt adatok formázása...")
test_dataset = test_dataset.map(format_text_for_test)

# ==============================================================================
# 4. PROFI KIÉRTÉKELŐ FÜGGVÉNY (SAMPLING + ROBUST PARSER)
# ==============================================================================

# Adatkinyerő
def extract_ingredients_robust(text):
    text = text.lower().strip()
    match = re.search(r"ingredients[:\s](.*?)steps[:\s]", text, re.DOTALL)
    if not match:
        match = re.search(r"ingredients[:\s](.*)$", text, re.DOTALL)
    if match:
        raw = match.group(1).strip()
        clean = raw.replace('\n', ',').replace('-', ',').replace('•', ',')
        return [i.strip() for i in clean.split(',') if len(i.strip()) > 1]
    return []

# Fő mérő függvény
def evaluate_final(model, tokenizer, dataset, num_samples=100):
    model.eval()
    test_subset = dataset.select(range(min(len(dataset), num_samples)))

    predictions = []
    references = []
    inputs_raw = []

    print(f"\nGenerálás {len(test_subset)} mintán (SAMPLING MÓDBAN)...")

    for row in tqdm(test_subset):
        inputs_raw.append(row["input_text"])
        references.append(row["target_text"])

        inputs = tokenizer(row["input_text"], return_tensors="pt", max_length=256, truncation=True).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=512,

                # --- LEGJOBB GENERÁLÁSI BEÁLLÍTÁSOK ---
                do_sample=True,          # Természetes szöveg
                top_k=50,                # 50 legjobb szó
                top_p=0.95,              # Nucleus sampling
                temperature=0.7,         # Kreativitás szabályozása (0.7 = kiegyensúlyozott)
                repetition_penalty=1.2,  # Ismétlés kerülése
                early_stopping=True
            )
        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append(pred_text)

    # Metrikák
    print("Metrikák számítása...")
    bertscore = evaluate.load("bertscore")
    rouge = evaluate.load("rouge")

    rouge_results = rouge.compute(predictions=predictions, references=references)
    bert_results = bertscore.compute(predictions=predictions, references=references, lang="en", device=device)
    avg_bert = np.mean(bert_results['f1'])

    # Ingredient Coverage
    precision_scores = []
    recall_scores = []
    valid_parsing_count = 0

    for i, pred in enumerate(predictions):
        gen_ings = extract_ingredients_robust(pred)
        target_ings = extract_ingredients_robust(references[i])

        if len(gen_ings) == 0:
            precision_scores.append(0.0)
            recall_scores.append(0.0)
            continue

        valid_parsing_count += 1
        hits = 0
        for gen_ing in gen_ings:
            for target_ing in target_ings:
                if gen_ing in target_ing or target_ing in gen_ing:
                    hits += 1
                    break
        prec = hits / len(gen_ings)
        rec = hits / len(target_ings) if len(target_ings) > 0 else 0
        precision_scores.append(prec)
        recall_scores.append(rec)

    avg_precision = np.mean(precision_scores) if precision_scores else 0.0
    avg_recall = np.mean(recall_scores) if recall_scores else 0.0

    # ROUGE érték kinyerése
    r1 = rouge_results['rouge1']
    if not isinstance(r1, float): r1 = r1.mid.fmeasure # Ha régi verzió

    print("\n" + "="*40)
    print("EREDMÉNYEK")
    print("="*40)
    print(f"Sikeres formátum: {valid_parsing_count} / {len(predictions)}")
    print("-" * 40)
    print(f"ROUGE-1:      {r1*100:.2f}%")
    print(f"BERTScore:    {avg_bert*100:.2f}%")
    print("-" * 40)
    print(f"Precision:    {avg_precision*100:.2f}%")
    print(f"Recall:       {avg_recall*100:.2f}%")
    print("="*40)

    # Mentés CSV-be
    out_file = os.path.join(os.path.dirname(MODEL_PATH), "final_evaluation_results.csv")
    pd.DataFrame({
        "Input": inputs_raw, "Target": references, "Generated": predictions
    }).to_csv(out_file, index=False)
    print(f"Eredmények mentve: {out_file}")

    # Minta
    print("\n[MINTA]")
    print(f"Bemenet: {inputs_raw[0]}")
    print(f"Kimenet: {predictions[0]}")

# ==============================================================================
# 5. FUTTATÁS
# ==============================================================================
evaluate_final(model, tokenizer, test_dataset, num_samples=100)